# Procurement Approval Copilot (LangGraph)

This notebook builds an educational workflow for:
1. Vendor quote analysis
2. Risk flagging
3. Summary drafting
4. Approval decision support


## Learning Objectives

- Understand how LangGraph state flows across procurement-review nodes
- See how each transition adds decision-support context
- Produce a final recommendation package for approvers


## Workflow Graph

```text
START
  -> analyze_vendor_quotes
  -> flag_procurement_risks
  -> draft_decision_summary
  -> build_approval_support
  -> END
```


## State Transition Guide

Transition 1 (`analyze_vendor_quotes` -> `flag_procurement_risks`):
- Adds normalized quote comparison (price, SLA, lead time, terms)

Transition 2 (`flag_procurement_risks` -> `draft_decision_summary`):
- Adds risk register with severity and rationale

Transition 3 (`draft_decision_summary` -> `build_approval_support`):
- Adds executive summary for non-technical reviewers

Transition 4 (`build_approval_support` -> END):
- Produces final recommendation, tradeoffs, and approval checklist


## Step 1 - Install and Import

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph'])
print('langgraph installed')


langgraph installed


In [2]:
from typing import Dict, List, TypedDict

from langgraph.graph import END, START, StateGraph

print('Imports ready')


E:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Imports ready


## Step 2 - Define Shared State

In [3]:
class ProcurementState(TypedDict):
    purchase_request: str
    vendor_quotes_raw: List[Dict[str, str]]
    quote_analysis: str
    risk_flags: str
    decision_summary: str
    approval_support: str

print('State schema initialized')


State schema initialized


## Step 3 - Sample Procurement Inputs

In [4]:
purchase_request = (
    'Need 250 managed laptops for remote engineering teams with 3-year support, '
    'delivery within 30 days, and strict security baseline requirements.'
)

vendor_quotes = [
    {
        'vendor': 'NorthBridge Tech',
        'unit_price_usd': '1280',
        'delivery_days': '21',
        'sla': '24x7 support, 4-hour critical response',
        'warranty_years': '3',
        'payment_terms': 'Net 30',
        'security_cert': 'ISO27001, SOC2 Type II',
    },
    {
        'vendor': 'Pioneer Systems',
        'unit_price_usd': '1195',
        'delivery_days': '35',
        'sla': 'Business hours support only',
        'warranty_years': '2',
        'payment_terms': '50% upfront, 50% on delivery',
        'security_cert': 'SOC2 Type I',
    },
    {
        'vendor': 'Summit Procurement Group',
        'unit_price_usd': '1315',
        'delivery_days': '18',
        'sla': '24x7 support, next-business-day replacement',
        'warranty_years': '4',
        'payment_terms': 'Net 45',
        'security_cert': 'ISO27001, SOC2 Type II, TAA-compliant supply chain',
    },
]

print('Sample request and quotes loaded')
print('Vendors:', len(vendor_quotes))


Sample request and quotes loaded
Vendors: 3


## Step 4 - Node 1: Vendor Quote Analysis

In [5]:
def analyze_vendor_quotes(state: ProcurementState) -> dict:
    quotes = state['vendor_quotes_raw']

    lines = []
    lines.append('Quote Analysis Table')
    lines.append('Request: ' + state['purchase_request'])

    for q in quotes:
        total_cost = int(q['unit_price_usd']) * 250
        line = (
            f"- {q['vendor']}: unit=${q['unit_price_usd']}, total=${total_cost}, "
            f"delivery={q['delivery_days']}d, warranty={q['warranty_years']}y, terms={q['payment_terms']}"
        )
        lines.append(line)

    lines.append('Baseline constraints: <=30 day delivery preferred, >=3 year warranty preferred, strong security certifications required.')

    result = '\n'.join(lines)
    print('analyze_vendor_quotes complete')
    return {'quote_analysis': result}


## Step 5 - Node 2: Risk Flagging

In [6]:
def flag_procurement_risks(state: ProcurementState) -> dict:
    risks = []

    for q in state['vendor_quotes_raw']:
        vendor = q['vendor']

        if int(q['delivery_days']) > 30:
            risks.append(f'HIGH: {vendor} delivery exceeds target timeline ({q['delivery_days']} days).')

        if int(q['warranty_years']) < 3:
            risks.append(f'MEDIUM: {vendor} warranty shorter than requested baseline ({q['warranty_years']} years).')

        if 'upfront' in q['payment_terms'].lower():
            risks.append(f'MEDIUM: {vendor} requires upfront payment, increases financial exposure.')

        certs = q['security_cert'].lower()
        if 'soc2 type ii' not in certs:
            risks.append(f'HIGH: {vendor} lacks SOC2 Type II evidence.')

        if '24x7' not in q['sla'].lower():
            risks.append(f'MEDIUM: {vendor} support SLA may not meet enterprise incident-response expectations.')

    if not risks:
        risks.append('No significant procurement risks detected from provided quotes.')

    text = 'Risk Flags\n' + '\n'.join('- ' + r for r in risks)
    print('flag_procurement_risks complete with', len(risks), 'flags')
    return {'risk_flags': text}


## Step 6 - Node 3: Summary Drafting

In [7]:
def draft_decision_summary(state: ProcurementState) -> dict:
    quotes = state['vendor_quotes_raw']

    best_delivery = min(quotes, key=lambda q: int(q['delivery_days']))
    best_price = min(quotes, key=lambda q: int(q['unit_price_usd']))
    best_warranty = max(quotes, key=lambda q: int(q['warranty_years']))

    summary = []
    summary.append('Decision Summary Draft')
    summary.append('Fastest delivery: ' + best_delivery['vendor'])
    summary.append('Lowest unit cost: ' + best_price['vendor'])
    summary.append('Strongest warranty: ' + best_warranty['vendor'])
    summary.append('')
    summary.append('Tradeoff narrative:')
    summary.append('Lowest cost option introduces SLA and assurance concerns; strongest governance option costs more but reduces operational and compliance risk.')

    text = '\n'.join(summary)
    print('draft_decision_summary complete')
    return {'decision_summary': text}


## Step 7 - Node 4: Approval Decision Support

In [8]:
def build_approval_support(state: ProcurementState) -> dict:
    recommendation = []
    recommendation.append('Approval Decision Support Package')
    recommendation.append('')
    recommendation.append('Recommended vendor: Summit Procurement Group')
    recommendation.append('Reason: balanced delivery speed, strongest warranty, robust security certifications, lower contractual payment risk.')
    recommendation.append('')
    recommendation.append('Approval Checklist:')
    recommendation.append('- Confirm budget tolerance for higher unit price')
    recommendation.append('- Confirm legal acceptance of terms and warranty wording')
    recommendation.append('- Validate SOC2 Type II and ISO certificates')
    recommendation.append('- Obtain stakeholder sign-off from IT security and finance')
    recommendation.append('')
    recommendation.append('If budget-constrained fallback is needed: NorthBridge Tech as secondary option, pending final commercial negotiation.')

    output = '\n'.join(recommendation)
    print('build_approval_support complete')
    return {'approval_support': output}


## Step 8 - Build LangGraph

In [9]:
builder = StateGraph(ProcurementState)
builder.add_node('analyze_vendor_quotes', analyze_vendor_quotes)
builder.add_node('flag_procurement_risks', flag_procurement_risks)
builder.add_node('draft_decision_summary', draft_decision_summary)
builder.add_node('build_approval_support', build_approval_support)

builder.add_edge(START, 'analyze_vendor_quotes')
builder.add_edge('analyze_vendor_quotes', 'flag_procurement_risks')
builder.add_edge('flag_procurement_risks', 'draft_decision_summary')
builder.add_edge('draft_decision_summary', 'build_approval_support')
builder.add_edge('build_approval_support', END)

app = builder.compile()
print('Workflow compiled')


Workflow compiled


## Step 9 - Execute Workflow

In [10]:
initial_state = {
    'purchase_request': purchase_request,
    'vendor_quotes_raw': vendor_quotes,
    'quote_analysis': '',
    'risk_flags': '',
    'decision_summary': '',
    'approval_support': '',
}

result = app.invoke(initial_state)
print('Run complete')


analyze_vendor_quotes complete
flag_procurement_risks complete with 5 flags
draft_decision_summary complete
build_approval_support complete
Run complete


## Step 10 - Review Outputs

In [11]:
print('=' * 90)
print('1) QUOTE ANALYSIS')
print('=' * 90)
print(result['quote_analysis'])

print('\n' + '=' * 90)
print('2) RISK FLAGS')
print('=' * 90)
print(result['risk_flags'])

print('\n' + '=' * 90)
print('3) DECISION SUMMARY')
print('=' * 90)
print(result['decision_summary'])

print('\n' + '=' * 90)
print('4) APPROVAL SUPPORT')
print('=' * 90)
print(result['approval_support'])


1) QUOTE ANALYSIS
Quote Analysis Table
Request: Need 250 managed laptops for remote engineering teams with 3-year support, delivery within 30 days, and strict security baseline requirements.
- NorthBridge Tech: unit=$1280, total=$320000, delivery=21d, warranty=3y, terms=Net 30
- Pioneer Systems: unit=$1195, total=$298750, delivery=35d, warranty=2y, terms=50% upfront, 50% on delivery
- Summit Procurement Group: unit=$1315, total=$328750, delivery=18d, warranty=4y, terms=Net 45
Baseline constraints: <=30 day delivery preferred, >=3 year warranty preferred, strong security certifications required.

2) RISK FLAGS
Risk Flags
- HIGH: Pioneer Systems delivery exceeds target timeline (35 days).
- MEDIUM: Pioneer Systems warranty shorter than requested baseline (2 years).
- MEDIUM: Pioneer Systems requires upfront payment, increases financial exposure.
- HIGH: Pioneer Systems lacks SOC2 Type II evidence.
- MEDIUM: Pioneer Systems support SLA may not meet enterprise incident-response expectation

## Educational Recap

State transition summary:
1. Analysis transition: converts raw quotes to comparable evidence.
2. Risk transition: converts evidence to explicit procurement risk flags.
3. Summary transition: converts risks into decision narrative for stakeholders.
4. Approval-support transition: converts narrative into action-ready approval package.
